# TP1 — Inverse Stage  
## Parameter Identification via Physics-Informed Neural Networks

This notebook implements the **inverse stage** of **Test Problem 1 (TP1)**.  
The objective is to **identify the unknown physical parameters** of a parametrized Poisson equation defined on a complex 3D geometry, by exploiting **Physics-Informed Neural Networks (PINNs)** and simulated IoT-like measurements.

This stage corresponds to the **Inverse Problem Submodule** of the Inference Engine described in the paper.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"
load_data_bound = ABS_PATH + "./files/data.csv"

fig_dir = "figures/"
loss_fig = ABS_PATH + fig_dir + "loss.png"
param_fig = ABS_PATH + fig_dir + "param.png"

params_save = ABS_PATH + "files/pred_parameters.csv"

Import of packages

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from model_acquisition import Blend2Pina

from pina.solvers.pinns import RBAPINN
from pina.operators import laplacian
from pina.callbacks import MetricTracker
from pina.geometry import CartesianDomain
from pina.model import ResidualFeedForward
from pina import LabelTensor, Trainer, Plotter
from pina.condition import Condition, Equation
from pytorch_lightning.callbacks import Callback, StochasticWeightAveraging
from pina.problem import SpatialProblem, InverseProblem

Set the double precision

In [ ]:
torch.set_default_dtype(torch.float64)

Definition of useful variables

In [ ]:
# Network variables
lear_rate = 1e-3
swa_lr = 1e-4
decay_rt = 1e-8

# Solver variables
epochs=3_000
batch=50
acc_str = 'gpu'

ipt_var = 3
out_var = 1
lay = 2
neur = 400

# Num points
int_points = 500
bound_points = 200

## Geometry management and loading of data

Here the notebook focuses on the interaction between the preprocessed Blender 3D model (See notebook *00_ProblemSettings.ipynb*) with the PINA Geometry module.
In addition. in this fase the simulated data is loaded.

In [ ]:
rock = Blend2Pina(LOAD_MODEL + model_name)

rock_int = rock.intern()
rock_bound = rock.boundary()

In [ ]:
df = pd.read_csv(load_data_bound, sep=";", index_col=0)

input_pts = df.iloc[:, :3].values
output_pts = df.iloc[:, -1].values

input_pts = LabelTensor(
    x=torch.tensor(input_pts, dtype=torch.float64),
    labels=['x', 'y', 'z']
)
output_pts = LabelTensor(
    x=torch.reshape(torch.tensor(output_pts, dtype=torch.float64), (output_pts.shape[0], 1)),
    labels=['u']
)

## Governing Physical Model

We consider the parametrized Poisson equation on the 3D domain $ \Omega \subset \mathbb{R}^3 $:


\begin{cases}
\Delta u(x,y,z;\mu) = -(\alpha^2 + \beta^2)\pi^2 \lambda x \cos(\alpha \pi y)\sin(\beta \pi z), & \text{in } \Omega, \\
u(x,y,z;\mu) = \lambda x \cos(\alpha \pi y)\sin(\beta \pi z), & \text{on } \partial \Omega,
\end{cases}

where the unknown parameter vector is

\begin{equation}
\mu = (\lambda, \alpha, \beta).
\end{equation}

### Inverse Problem Formulation

The inverse problem consists of identifying the parameter vector $ \mu $ from a set of observed data.

Let $ \mathcal{D} = \{(x_i, u_i)\}_{i=1}^{N_d} $ denote a set of measurements collected on the boundary $ \partial\Omega $, where the values $ u_i $ are generated synthetically and emulate IoT sensor observations.

The goal is to recover $ \mu $ such that the reconstructed solution:
- satisfies the governing PDE,
- matches the observed data,
- respects the boundary conditions.

In the next block, the problem is defined.


In [ ]:
class RockPoisson(SpatialProblem, InverseProblem):

    input_variables = ['x','y','z']
    output_variables = ['u']
    spatial_domain = rock_int
    unknown_parameter_domain = CartesianDomain(
        {
            'lambda' : [0, 1],
            'alpha' : [0, 1],
            'beta' : [0, 1]
        }
    )

    # Residual
    @staticmethod
    def residual(input_, output_, params_):
        lap_u = laplacian(output_=output_, input_=input_, components=['u'], d=['x', 'y', 'z'])
        force_term = - (params_['alpha']**2 + params_['beta']**2) * (
            torch.pi**2 * params_['lambda'] * input_.extract('x') * 
            torch.cos(params_['alpha'] * torch.pi * input_.extract('y')) * 
            torch.sin(params_['beta'] * torch.pi * input_.extract('z'))
        )
        return lap_u - force_term
    
    def boundary(input_, output_, params_):
        ref = (
            params_['lambda'] * input_.extract('x') *
            torch.cos(params_['alpha'] * torch.pi * input_.extract('y')) * 
            torch.sin(params_['beta'] * torch.pi * input_.extract('z'))
        )
        return output_.extract('u') - ref
    
    conditions = {
        'Omega' : Condition(
            location=rock_int,
            equation=Equation(residual)
        ),
        'Gamma' : Condition(
            location=rock_bound,
            equation=Equation(boundary)
        ),
        'data' : Condition(
            input_points=input_pts.extract(['x','y','z']),
            output_points=output_pts
        )
    }

After the definition of the class, the sampling of collocation and boundary points is performed.

In [ ]:
problem = RockPoisson()

problem.discretise_domain(
    n=int_points,
    mode='random',
    locations=['Omega']
)

problem.discretise_domain(
    n=bound_points,
    mode='random',
    locations=['Gamma']
)

## Physics-Informed Neural Network (PINN)

A Physics-Informed Neural Network is employed to approximate the solution

\begin{equation}
u(x,y,z;\mu) \approx u_{\theta}(x,y,z),
\end{equation}

where $ u_{\theta} $ is a neural network parameterized by weights $ \theta $.

In the inverse setting, the unknown physical parameters $ \mu $ are treated as **trainable variables** and optimized jointly with the network weights.

### PINN Loss Function

The training of the PINN is driven by a composite loss function:

\begin{equation}
\mathcal{L} =
\mathcal{L}_{\text{PDE}} +
\mathcal{L}_{\text{BC}} +
\mathcal{L}_{\text{data}},
\end{equation}

where:

- **PDE residual loss**

\begin{equation}
\mathcal{L}_{\text{PDE}} =
\frac{1}{N_{\Omega}}
\sum_{i=1}^{N_{\Omega}}
\left\|
\Delta u_{\theta}(x_i) + (\alpha^2 + \beta^2)\pi^2 \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Boundary condition loss**

\begin{equation}
\mathcal{L}_{\text{BC}} =
\frac{1}{N_{\partial\Omega}}
\sum_{i=1}^{N_{\partial\Omega}}
\left\|
u_{\theta}(x_i) - \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Data loss**

\begin{equation}
\mathcal{L}_{\text{data}} =
\frac{1}{N_d}
\sum_{i=1}^{N_d}
\left\|
u_{\theta}(x_i) - u_i
\right\|^2.
\end{equation}

In [ ]:
class HardMLP(torch.nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__()
        self.layers = ResidualFeedForward(*args, **kwargs)

    def forward(self, x):
        return  self.layers(x)

Definition of the class to save the parameters during the training phase.

In [ ]:
# Directory to save the updating of parameters during the training
tmp_dir = ABS_PATH + "rock_poisson_inverse"

class SaveParameters(Callback):
    """
        Callback to save parameters.
    """
    def on_train_epoch_end(self, trainer, _):
        if trainer.current_epoch % 100 == 99:
            torch.save(
                trainer.solver.problem.unknown_parameters,
                '{}/parameters_epoch{}'.format(tmp_dir, trainer.current_epoch)
            )

Inizialization of the NN model, the PINN solver, the Trainer. Then, the training phase starts.

In [ ]:
# Model
model = HardMLP(
    input_dimensions=ipt_var,
    output_dimensions=out_var,
    n_layers=lay,
    inner_size=neur
)
# Solver
pinn=RBAPINN(
    problem=problem,
    model=model,
    optimizer_kwargs={
        'lr' : lear_rate,
        'weight_decay' : decay_rt
    },
)

# Trainer
trainer=Trainer(
    solver=pinn,
    max_epochs=epochs,
    batch_size=None,
    accelerator=acc_str,
    precision='64-true',
    callbacks=[SaveParameters(), MetricTracker(), StochasticWeightAveraging(swa_lrs=swa_lr)]
)

# Training phase
trainer.train()

Plot of the losses related to the training process.

In [ ]:
my_pl = Plotter()

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Omega_loss'],
    label='Omega_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Gamma_loss'],
    label='Gamma_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['data_loss'],
    label='data_loss',
    logy=True
)

plt.savefig(loss_fig, transparent=True)
plt.show()

## Convergence Monitoring

During training, the evolution of the inferred parameters $ \mu = (\lambda, \alpha, \beta) $ is monitored and compared against the reference values used to generate the synthetic data.

This analysis provides a direct assessment of the accuracy and stability of the inverse identification process.

In [ ]:
epochs_saved = range(99, epochs, 100)
parameters = torch.empty(
    size=(int(epochs/100), 3)
)

for i, epoch in enumerate(epochs_saved):
    params_torch = torch.load('{}/parameters_epoch{}'.format(tmp_dir, epoch))
    for e, var in enumerate(pinn.problem.unknown_variables):
        parameters[i, e] = params_torch[var].data

pred_alpha, pred_beta, pred_lambda = parameters[-1, :]

plt.close()
plt.plot(epochs_saved, parameters[:, 2], label='lambda', marker='o')
plt.plot(epochs_saved, parameters[:, 0], label='alpha', marker='s')
plt.plot(epochs_saved, parameters[:, 1], label='beta', marker='^')
plt.ylim(0, 1)
plt.grid()
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Parameters")
plt.savefig(param_fig, transparent=True)
plt.show()

Relative errors w.r.t. the exact parameters.

In [ ]:
par_lambda = torch.tensor(.1)
par_alpha = torch.tensor(.2)
par_beta = torch.tensor(.5)

err_rel_lambda = torch.norm(pred_lambda-par_lambda)/torch.norm(par_lambda)
err_rel_alpha = torch.norm(pred_alpha-par_alpha)/torch.norm(par_alpha)
err_rel_beta = torch.norm(pred_beta-par_beta)/torch.norm(par_beta)

print("RELATIVE ERRORS")
print(f"lambda: {err_rel_lambda.item(): .2e}")
print(f"alpha: {err_rel_alpha.item(): .2e}")
print(f"beta: {err_rel_beta.item(): .2e}")

Saving the parameters and the model.

In [ ]:
df = pd.DataFrame(
    data=[[pred_lambda.item()], [pred_alpha.item()], [pred_beta.item()]],
    columns=["predictions"],
    index=["lambda", "alpha", "beta"]
)

df.to_csv(params_save, sep=";")

In [ ]:
torch.save(model, ABS_PATH + f"models/model_L{lay}_N{neur}_EP{epochs}.pth")

## Inverse Stage Outputs

At the end of the inverse stage, the following results are obtained:

- estimated physical parameters $ \mu = (\lambda, \alpha, \beta) $,
- trained PINN model consistent with physics and data,
- convergence histories for parameters and loss terms.

The inferred parameters are subsequently used in the **online stage** to compute fast reduced-order simulations.